# EX04 - Attribute Grammars and LarkAG

## Grammar Engineering

## MEI/2026-27

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved with pen and paper and implemented with an AG processor for Python, [LarkAG](https://larkag.epl.di.uminho.pt).

# Processing Attribute Grammars

- The semantics defined by an attribute grammar (AG) can be implemented manually using syntax-directed translations or traversals of abstract syntax trees (ASTs)
  - Cumbersome and error-prone

- However, it is also possible to process them directly and generate a translator that applies the attribute evaluation rules

- Allows us to just specify a context-free grammar (CFG), the attributes associated with each syntactic variable, and their evaluation rules

- The order of traversal and attribute computation is then determined by the AG processor

## AGs in Python and `Lark`

- `Lark` does not support AGs directly

- `LarkAG` is a (prototype) extension that supports AGs on top of `Lark` grammars, and translates them into `Lark` AST traversals

- It uses the `Lark` `Interperter` class, which allows full control on the order of traversal, essential to process syntesized and inherited attributes

- `Lark`'s `Interpreter` gives the implementation explicit control over tree traversal

- This is a prototype tool with some limitations, but highlights the main strengths of thinking at the AG level:
  - Decoupling semantics from the parser and implementation details

⚠️ Note: the `LarkAG` version in PyPI is not up-to-date, so you should manually install the latest version from a forked [repository](https://github.com/nmacedo/LarkAG.git)

In [1]:
!pip install --no-cache-dir --force-reinstall "git+https://github.com/nmacedo/LarkAG.git@ag-soundness-fixes"

  Cloning https://github.com/nmacedo/LarkAG.git (to revision ag-soundness-fixes) to /tmp/pip-req-build-b085ehyi
  Running command git clone --filter=blob:none --quiet https://github.com/nmacedo/LarkAG.git /tmp/pip-req-build-b085ehyi
  Running command git checkout -b ag-soundness-fixes --track origin/ag-soundness-fixes
  Trocou para um novo ramo 'ag-soundness-fixes'
  ramo 'ag-soundness-fixes' configurado para monitorizar 'origin/ag-soundness-fixes'.
  Resolved https://github.com/nmacedo/LarkAG.git to commit cf93ccb318f1827c10b3999d01b3e3de0962eeeb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lark-ag: filename=lark_ag-1.0.3-py2.py3-none-any.whl size=14524 sha256=d42f5d2a4d2141a0e40e2aeebcaa618f300969ae06a5ae3f769838d1e87988e4
  Stored in directory: /tmp/pip-ephem-wheel-cache-x2i328ue/wheels/d7/14/df/516e5ccc6aef58bb567a7bce695823ff1e4445ec7477e76987
Successfully built lark-ag
  

## `LarkAG` basics

- Start by defining the attributes of each syntactic variable, either synthesized (`SA`) or inherited (`IA`)

- Write a CFG using `Lark` notation, but each production can be interleaved with 3 sets of rules:
  - evaluation rules `ER`, declaritevly determing the value of the attributes from other attributes
  - context conditions `CC`, determine necessary conditions for evaluation, fail otherwise
  - translation rules `TR`, further transformations to be triggered beyond attribute evaluation, arbitrary code

- Attributes are referred to as `symbol.attribute`. When the same non-terminal occurs more than once in a production rule, use indexes (starting at 1) to select the occurrence; the symbol on the left-hand side counts as the first one:
  - in `expr : expr ADD term`, `expr[1]` is the node being defined and `expr[2]` is its child

- A terminal is referred to by its name and is a string, e.g. `str(ID)`, `int(NUMBER)`

- For repeated children such as `item*`, `$item` is the number of occurrences and `@item` the list of occurrences, e.g. `sum([i.price for i in @item])`

- A failed context condition raises a `ContextConditionError` with the message given after `->`

- The entry point of the grammar must be the rule `start`

## ✏️ `LarkAG`: Evaluate arithmetic expressions with declared variables

Let's recall the running example of arithmetic expressions with declared variables, and its encoding as an AG

- `A(start) = { (val, synthesized) }`
- `A(decls) = { (val, synthesized) }`
- `A(decl) = { (val, synthesized), (env, inherited) }`
- `A(expr) = { (val, synthesized), (env, inherited) }`
- `A(term) = { (val, synthesized), (env, inherited) }`
- `A(factor) = { (val, synthesized), (env, inherited) }`
- `A(literal) = { (val, synthesized), (env, inherited) }`

| Production          | Evaluation rules    | Context conditions |
|----------------------|-----------------------|--|
| `prog → decls expr`           | `expr.env = decls.val` | |
|                      | `prog.val = expr.val` | |
| `decls₀ → decls₁ decl`        | `decl.env = decls₁.val` | |
|                      | `decls₀.val = decl.val` | |
| `decls → ε`             | `decls.val = {}` | |
| `decl → let ID = expr ;`   | `expr.env = decl.env` | |
|                      | `decl.val = extend(decl.env, ID.lexval → expr.val)` | `ID.lexval ∉ decl.env` |
| `expr₀ → expr₁ + term`        | `expr₁.env = expr₀.env` | |
|                      | `term.env = expr₀.env` | |
|                      | `expr₀.val = expr₁.val + term.val` | |
| `expr → term`              | `term.env = expr.env` | |
|                      | `expr.val = term.val` | |
| `term₀ → term₁ * factor`        | `term₁.env = term₀.env` | |
|                      | `factor.env = term₀.env` | |
|                      | `term₀.val = term₁.val * factor.val` | |
| `term → factor`              | `factor.env = term.env` | |
|                      | `term.val = factor.val` | |
| `factor → ( expr )`          | `expr.env = factor.env` | |
|                      | `factor.val = expr.val` | |
| `factor → ID`             | `factor.val = factor.env(ID.lexval)` | `ID.lexval ∈ factor.env` |
| `factor → INT`	           | `factor.val = INT.lexval` | |
 |

In [ ]:
grammar = r"""
SA(start)   { val : int }

SA(decls)   { val : dict[str, int] }

SA(decl)    { val : dict[str, int] }
IA(decl)    { env : dict[str, int] }

SA(expr)    { val : int }
IA(expr)    { env : dict[str, int] }

SA(term)    { val : int }
IA(term)    { env : dict[str, int] }

SA(factor)  { val : int }
IA(factor)  { env : dict[str, int] }

start : decls expr
ER {
  expr.env = decls.val;
  start.val = expr.val;
}

decls : decls decl
ER {
  decl.env = decls[2].val;
  decls[1].val = decl.val;
}
|
ER {
  decls.val = {};
}

decl : "let" ID "=" expr ";"
ER {
  expr.env = decl.env;
  decl.val = decl.env | { str(ID): expr.val };
}
CC {
  str(ID) not in decl.env -> "Variable already declared";
}

expr : expr "+" term
ER {
  expr[2].env = expr[1].env;
  term.env = expr[1].env;
  expr[1].val = expr[2].val + term.val;
}
| term
ER {
  term.env = expr.env;
  expr.val = term.val;
}

term : term "*" factor
ER {
  term[2].env = term[1].env;
  factor.env = term[1].env;
  term[1].val = term[2].val * factor.val;
}
| factor
ER {
  factor.env = term.env;
  term.val = factor.val;
}

factor : "(" expr ")"
ER {
  expr.env = factor.env;
  factor.val = expr.val;
}
| ID
ER {
  factor.val = factor.env.get(str(ID), 0);
}
CC {
  str(ID) in factor.env -> "Undeclared variable";
}
| INT
ER {
  factor.val = int(INT);
}

ID : /[a-z]+/
INT : /[0-9]+/
"""

In [ ]:
from lark_ag import Lark_AG, ContextConditionError

In [ ]:
parser = Lark_AG(grammar)

In [ ]:
try:
    res = parser.process("let x = 20; let y = x*2; x + y")
    print(res.val)
except ContextConditionError as e:
    print(f"Error: {e}")

60


In [ ]:
try:
    res = parser.process("let x = 20; let y = x*2; x + y + z")
    print(res.val)
except ContextConditionError as e:
    print(f"Error: {e}")

Error: Undeclared variable


In [ ]:
try:
    res = parser.process("let x = 20; let x = 1; x")
    print(res.val)
except ContextConditionError as e:
    print(f"Error: {e}")

Error: Variable already declared


In [ ]:
import doctest

# Exercise 1

Consider a list of values with mixed types, whose sum of the numeric values must be positive. For instance, the following is a valid list:

```
( comentario, -2, nota, alto, 5, baixo, 4, 12, 4 )
```

but the following is invalid:
```
( comentario, -2, nota, alto, 5, baixo, 4, -12, 4 )
```


## Exercise 1.1

Write an AG for finding the **average** value of a mixed type list. Write the CFG, the attributes, their evaluation rules, and context conditions.

## Exercise 1.2

Implement this AG using `LarkAG`.

In [ ]:
grammar = """

"""

In [ ]:
def mixed_lists(lst):
    """
    >>> mixed_lists("( 1, 2, 3, 2, 1 )")
    1.8
    >>> mixed_lists("( 1, -2, -3, 2, 1 )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    >>> mixed_lists("( comentario, -2, nota, alto, 5, baixo, 4, 12, 4 )")
    4.6
    >>> mixed_lists("( comentario, -2, nota, alto, 5, baixo, 4, -12, 4 )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    >>> mixed_lists("( 0, 4 )")
    2.0
    >>> mixed_lists("( comentario, nota )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    """

    parser = lark_ag.Lark_AG(grammar)
    res = parser.process(lst)
    return None

doctest.run_docstring_examples(mixed_lists, globals())

## Exercise 1.3

- Consider now that there are two special keywords that can occur in the list: `p`, after which all numerical values must be positive, and `n` after which all numerical values must be negative

- Rewrite the AG and its implementation in `LarkAG`, still for calculating the average value, but checking the validity of the numerical values

In [ ]:
grammar = """

"""

In [ ]:
def mixed_lists(lst):
    """
    >>> mixed_lists("( p, 1, 2, 3, 2, 1 )")
    1.8
    >>> mixed_lists("( n, -1, -2, -5, -4, p, n, p, 1, 30, 2 )")
    3.0
    >>> mixed_lists("( p, 1, -2, -3, -4, p, 1, 30, 2 )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Wrong signal element
    >>> mixed_lists("( n, -1, -200, -3, -4, p, 1, 30, 2 )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    """

    parser = Lark_AG(grammar)
    res = parser.process(lst)
    return None

doctest.run_docstring_examples(mixed_lists, globals())

## Exercise 1.4

- We now want to implement contextual addition for these lists: whatever numerical values appear between special keyword `start` and `stop` is processed, anything else is ignored; a list may contain multiple `start` and `stop` segments

- Adapt the AG to represent this contextual addition, and implement it in `LarkAG`

In [ ]:
grammar = """

"""

In [ ]:
def mixed_lists(lst):
    """
    >>> mixed_lists("( start, 1, 2, 3, 2, 1, stop )")
    1.8
    >>> mixed_lists("( -6, -8, start, 1, 2, 3, 2, 1, stop, -8, -9 )")
    1.8
    >>> mixed_lists("( -6, -8, start, 1, 2, stop, 2, 6, start, 3, 2, 1, stop, -8, -9 )")
    1.8
    >>> mixed_lists("( start, -9, 2, -3, 2, 1, stop )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    >>> mixed_lists("( start, 0, 4, stop )")
    2.0
    >>> mixed_lists("( 5, 6 )")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Non-positive sum
    """

    parser = lark_ag.Lark_AG(grammar)
    res = parser.process(lst)
    return None

doctest.run_docstring_examples(mixed_lists, globals())

# Exercise 2

Let's revisit the language for first-order logic formulas with the following syntax:

- Predicates $A$, $B$, $C$, ...
- Variables $a$, $b$, $c$, ...
- Constants false $⊥$ and true $⊤$
- Predicate application $A(a_1, \ldots, a_n)$ if $A$ has arity $n$
- Conjunction $f_1 ∧ f_2$,
- Disjunction $f_1 ∨ f_2$,
- Negation $¬ f_1$,
- Implication $f_1 → f_2$
- Equivalence $f_1 ↔︎ f_2$
- Universal quantification $∀ a . f_1$
- Existential quantification $∃ a . f_1$

## Exercise 2.1

Write an AG for finding all occurrences of predicates within a formula, validating whether predicate and variable occurrenceses are valid. Write the CFG, the attributes, their evaluation rules, and context conditions.

## Exercise 2.2

Implement this AG using `LarkAG`.

In [ ]:
grammar = """

"""

In [ ]:
def predicates(formula):
    """
    >>> predicates("P : 1 Q : 2  ∀ x . ∃ y . P(x) ∧ Q(x, y) → ¬ P(y)")
    ['P', 'Q', 'P']
    >>> predicates("P : 1  ∀ a . P(a) ∧ (∀ x . P(x)) ∨ ⊤")
    ['P', 'P']
    >>> predicates("P : 1  P(x)")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Unbound variable
    >>> predicates("P : 1  ∀ x . P(x) ∧ P(y)")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Unbound variable
    >>> predicates("P : 1  ∀ x . Q(x)")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Undeclared predicate
    >>> predicates("Q : 2  ∀ x . Q(x)")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Wrong number of arguments
    >>> predicates("P : 1 P : 2  ⊤")
    Traceback (most recent call last):
        ...
    lark_ag.errors.ContextConditionError: Predicate already declared
    """

    res = Lark_AG(grammar).process(formula)
    return None

doctest.run_docstring_examples(predicates, globals())

# Exercise 3

Consider a language to represent nested lists of strings, where each sub-list has a header. For instance, one such list is as follows:
```
(Introduction, Background (Formalism, Tools), Approach (Overview, Implementation (Architecture, Evaluation), Discussion), Conclusion)
```

## Exercise 3.1

Write an AG that translates such lists into a numbered list format with indentation, such as the following for the example list shown above.
```
1. Introduction
2. Background
  2.1. Formalism
  2.2. Tools
3. Approach
  3.1. Overview
  3.2. Implementation
    3.2.1. Architecture
    3.2.2. Evaluation
  3.3. Discussion
4. Conclusion
```

## Exercise 3.2

Implement this AG using `LarkAG`.

In [ ]:
grammar = """

"""

In [ ]:
def numbered_list(lst):
    """
    >>> numbered_list('(Introduction, Background (Formalism, Tools), Approach (Overview, Implementation (Architecture, Evaluation), Discussion), Conclusion)')
    1. Introduction
    2. Background
      2.1. Formalism
      2.2. Tools
    3. Approach
      3.1. Overview
      3.2. Implementation
        3.2.1. Architecture
        3.2.2. Evaluation
      3.3. Discussion
    4. Conclusion
    """

    Lark_AG(grammar).process(lst)

doctest.run_docstring_examples(numbered_list, globals())

-- nmm, 2026-10-22